### 1. Data Ingestion & Initial Inspection
We loaded the dataset to inspect its structure, verifying the total number of records (rows) and attributes (columns). 
This initial check helps identify feature types (numerical, categorical, and temporal) as well as the immediate presence of missing values.
Below we display the first 5 rows to understand the data layout.

In [ ]:
import pandas as pd
import numpy as np
trainFile = "/Users/matilderibeiro/Desktop/NOVA/Machine Learning/ML_Grupo18/data/train.csv"
df = pd.read_csv(trainFile)

print("Dataset Dimention (Columns, Rows):", df.shape)

print("\n--- First 5 Rows ---")
df.head()

### 1.1. Feature Overview & Data Types
The following code snippet displays the full list of features, their respective non-null counts, and inferred data types.

In [ ]:
print("\n--- Datatype Information ---")
print(df.info())

### 1.2. Analysis of Duplicates and Customer Repeat Purchases

To ensure data integrity and understand the granularity of our dataset, we performed two key checks:
1. **Full Row Duplicates (`df.duplicated()`):** We checked for exact duplicate entries across all columns in the df dataset. Finding zero full duplicates confirms that every record represents a unique invoice/transaction event.
2. **Granularity & Customer Repetition (`customer_id`):** We examined the frequency of transactions per customer (`customer_id`). Multiple records sharing the same `customer_id` do not indicate data errors, but rather reflect returning customers who made multiple purchases over time.

Analyzing the distribution of purchases per customer helps validate the transactional nature of the dataset and aligns directly with our modeling objective of predicting customer repeat purchases.

In [ ]:
duplicate_rows = df.duplicated().sum()
print("duplicates ignoring the index:", duplicate_rows)

per_customer = df.groupby("customer_id").size()
print(f"\nsurplus rows on 'customer_id': {df.duplicated(subset='customer_id').sum()}")
print(f"rows sharing a 'customer_id': {per_customer[per_customer > 1].sum()} across {(per_customer > 1).sum()} customers")
print("records per customer:", per_customer.value_counts().to_dict())

Count the total number of rows belonging to customers with more than 1 purchase (involved rows), and determine how many unique customers this group represents.


In [ ]:
print(f"rows sharing a 'customer_id': {per_customer[per_customer > 1].sum()} across {(per_customer > 1).sum()} customers")

### 1.3. Missing Value Analysis
To assess overall data completeness in the raw dataset, we generated a high-level summary of missing values (`NaN`).

This overview quantifies:
* **Rows with at least one missing value** (total count and percentage of the dataset).
* **Columns containing missing values** out of the total feature set.
* **Total missing cells** across the entire data frame.
* **The feature with the highest count of missing values** (`worst column`).

This summary establishes an initial baseline of data quality prior to any cleaning or imputation steps.

In [ ]:
rows_with_null = df.isna().any(axis=1)
per_column = df.isna().sum()

pd.Series({
    "rows with a null": f"{rows_with_null.sum():,} of {len(df):,} ({100 * rows_with_null.mean():.2f}%)",
    "columns with a null": f"{(per_column > 0).sum()} of {df.shape[1]}",
    "null cells": f"{df.isna().sum().sum():,}",
    "worst column": f"{per_column.idxmax()} ({per_column.max()} nulls)",
}).to_frame("missing values as the file arrives")


--- NA Values per column ---
Total a Pagar                       65
Total Bruto                         65
Total Liquido                       65
Total Imp_                          65
Total Portes                        65
Total Desc_ Global                  65
Total Desc_ Linha                   65
Custo                               65
N_ Detalhes                         65
invoice_count_current_purchase      65
repeat_purchase_90d                  0
days_since_previous_purchase       861
customer_tenure_days                65
prior_purchase_count                65
prior_total_spend                   65
prior_mean_purchase_value          862
prior_std_purchase_value          1113
prior_min_purchase_value           668
prior_max_purchase_value           668
prior_mean_gap_days               1308
prior_std_gap_days                1478
purchase_count_30d                  65
spend_30d                           65
purchase_count_90d                  65
spend_90d                         

### 1.4. Identification and Handling of Domain Inconsistencies (Negative Values)

In domain-specific feature engineering, certain numerical attributes—such as days elapsed, monetary amounts, and purchase values—must be non-negative by definition.

We identified columns where negative values represent domain inconsistencies or data anomalies:
* `days_since_previous_purchase`
* `prior_mean_purchase_value`
* `prior_mean_gap_days`
* `spend_365d`

To evaluate the impact of these anomalies, we temporarily replace negative values with `NaN` using `.mask(< 0)` and quantify how this transformation affects the total count of missing rows and missing cells across the dataset.


In [ ]:
# Define numerical columns whose domain strictly forbids negative values
non_negative = [
    "days_since_previous_purchase", 
    "prior_mean_purchase_value", 
    "prior_mean_gap_days", 
    "spend_365d"
]

# Convert negative values in these columns to NaN to evaluate anomalies
df_cleaned = df.copy()
df_cleaned[non_negative] = df_cleaned[non_negative].mask(df_cleaned[non_negative] < 0)

# Calculate the impact on missing value counts
rows_before = df.isna().any(axis=1).sum()
rows_after = df_cleaned.isna().any(axis=1).sum()
cells_before = df.isna().sum().sum()
cells_after = df_cleaned.isna().sum().sum()

print(f"\nrows with a null: {rows_before:,} -> {rows_after:,} ({100 * rows_after / len(df):.2f}% of {len(df):,})")
print(f"null cells:       {cells_before:,} -> {cells_after:,}")

COMPLETAR DEPOIS DE CORRER O CODIGO!!!!!!!!!!!!!!!!!!!!!!

Identificação de Dados Inválidos: No contexto do negócio, métricas como o número de dias desde a última compra (days_since_previous_purchase), o valor médio de compras anteriores (prior_mean_purchase_value), o intervalo médio entre compras (prior_mean_gap_days) e o gasto acumulado nos últimos 365 dias (spend_365d) não podem assumir valores inferiores a zero.   Tratamento por Invalidação: Os registos com valores negativos nestas colunas representam anomalias na recolha de dados. Em vez de apagar as linhas inteiras ou ignorar os erros, converteu-se esses valores específicos em NaN (missing values) para serem devidamente imputados na fase de pré-processamento.   Impacto no Dataset: Após a conversão, o número de linhas com pelo menos um valor omisso subiu ligeiramente de 2 605 para 2 643 (40,45% do total de observações), e o número total de células nulas passou de 9 818 para 9 881.

### 2. Data Cleaning: Dropping Corrupted & Fully Missing Rows
We identified 65 rows ($\approx 1\%$ of the dataset) with missing values across all transaction attributes. Because they provide no predictive signal, these corrupted records were removed from the training dataset.

In [6]:
df_clean = df.dropna(subset=['Total a Pagar']).copy()

### 2.1 Verification of Missing Value Removal
To confirm the success of the data cleaning step, we re-evaluated the count of missing values (`NaN`) in the `'Total a Pagar'` column within the cleaned dataset (`df_clean`). 

As expected, the output confirms **0 missing values**, validating that all corrupted rows were successfully removed before moving forward to the feature engineering phase.

In [ ]:
# 1. Verificar diretamente se ainda existem valores nulos nessa coluna (deve retornar 0)
print("NA Values in 'Total a Pagar':", df_clean['Total a Pagar'].isnull().sum())

print(f"Rows before: {len(df)} | Rows after: {len(df_clean)} | Removed rows: {len(df) - len(df_clean)}")

NA Values in 'Total a Pagar': 0


### 2.2. Feature Type Separation & Exclusions
To prepare the dataset for model preprocessing and feature encoding, we systematically divide our attributes into distinct functional groups:

1. **Excluded Features (`identifiers` & `target`):** 
   * Primary transaction identifiers (`ID`) and customer keys (`customer_id`) are excluded to prevent data leakage and memorization (*overfitting*).
   * Raw date strings (`purchase_date`) are set aside after extracting engineered temporal features.
   * The target variable (`repeat_purchase_90d`) is isolated from the predictors ($X$).

2. **Numeric Features (`numeric`):** 
   * Dynamically selected continuous and discrete numerical indicators (e.g., spending metrics, historical purchase counts, and temporal gaps) that will undergo missing value imputation and scaling.

3. **Categorical Features (`categorical`):** 
   * Dynamically identified non-numeric attributes (e.g., payment types, sales agents, zones, warehouse locations) designated for categorical encoding (*One-Hot / Frequency Encoding*).

This automated partitioning ensures code scalability and seamless integration with `ColumnTransformer` pipelines.


In [ ]:
# Identifiers and dates that MUST NOT be used for model training
identifiers = ["ID", "customer_id", "purchase_date"]
target = "repeat_purchase_90d"

# Automatic separation between numerical and categorical variables
numeric = [
    c for c in df.columns 
    if c not in identifiers + [target] and pd.api.types.is_numeric_dtype(df[c])
]

categorical = [
    c for c in df.columns 
    if c not in identifiers + [target] + numeric
]


Tratamento de Valores Ausentes Estruturais (MNAR)

Valores ausentes estruturais -> A ausencia de dados em variáveis de histórico de compras não é aleatória (Missing Not At Random - MNAR): devem-se a Novos Clientes / Primeira Compra: As variáveis days_since_previous_purchase, prior_mean_purchase_value, prior_mean_gap_days e prior_std_gap_days têm taxas elevadas de nulos (ex.: 861 a 1478 falhas). Clientes que efetuaram a sua primeira compra ou que têm apenas 1 compra anterior não possuem intervalo de tempo (gap) nem desvio-padrão (std) para calcular.

In [9]:
#Criar uma flag binária para novos clientes (Primeira Compra)
df_clean['is_first_purchase'] = df_clean['days_since_previous_purchase'].isnull().astype(int)

In [10]:
#Imputação por 0 para colunas de variação/desvio/médias que dependem de múltiplas compras
#Se o cliente só tem 1 compra anterior ou nenhuma, o desvio/intervalo é 0
zero_impute_cols = [
    'prior_std_purchase_value', 
    'prior_std_gap_days', 
    'prior_mean_gap_days'
]
df_clean[zero_impute_cols] = df_clean[zero_impute_cols].fillna(0)

In [11]:
#Imputação por valor indicador (-1) para contagens de tempo e médias de novos clientes
#O valor -1 assinala explicitamente ao algoritmo que não existe histórico anterior
indicator_impute_cols = [
    'days_since_previous_purchase',
    'prior_mean_purchase_value',
    'prior_min_purchase_value',
    'prior_max_purchase_value'
]
df_clean[indicator_impute_cols] = df_clean[indicator_impute_cols].fillna(-1)

In [12]:
#Validação da Flag de Novos Clientes
novos_clientes = df_clean['is_first_purchase'].sum()
print(f"[2] Flag 'is_first_purchase': {novos_clientes} novos clientes identificados ({(novos_clientes/len(df_clean))*100:.1f}%)")

[2] Flag 'is_first_purchase': 796 novos clientes identificados (12.3%)


In [14]:
# Validação das Imputações por -1
contagem_menos_um = (df_clean[indicator_impute_cols] == -1).sum().to_dict()
print(f"-> Valoração com -1 aplicada por coluna: {contagem_menos_um}")

-> Valoração com -1 aplicada por coluna: {'days_since_previous_purchase': 796, 'prior_mean_purchase_value': 797, 'prior_min_purchase_value': 603, 'prior_max_purchase_value': 603}


In [15]:
#Variáveis Temporais:
#purchase_date está registada como texto (str). Temos de convertê-la para datetime (pd.to_datetime) 
#para extrair atributos temporais úteis (ano, mês, dia da semana, trimestre)
# Converter para datetime
df_clean['purchase_date'] = pd.to_datetime(df_clean['purchase_date'])

O comportamento de compra dos clientes varia drasticamente dependendo do momento no tempo. Ao descompor a data, permites ao modelo aprender padrões específicos:

Sazonalidade e Meses (purchase_month / purchase_quarter):

Existem meses de pico de vendas (ex.: Black Friday em novembro, compras de Natal em dezembro ou épocas de férias em agosto).

Exemplo real no teu dataset: Nos dados do projeto, compras efetuadas em maio apresentam uma taxa de recompra a 90 dias de 72.4%, enquanto em março caem para 43.8%.

Comportamento Semanal (purchase_dayofweek / is_weekend):

O perfil de compras B2B (empresas) ou B2C (consumidores finais) muda entre dias úteis e fins de semana.

Exemplo real no teu dataset: As compras efetuadas em dias úteis (is_weekend=0) têm uma probabilidade de recompra de 64.4%, enquanto nos fins de semana (is_weekend=1) caem para 51.5%.

In [16]:
# Extração de atributos temporais relevantes
df_clean['purchase_year'] = df_clean['purchase_date'].dt.year
df_clean['purchase_month'] = df_clean['purchase_date'].dt.month
df_clean['purchase_day'] = df_clean['purchase_date'].dt.day
df_clean['purchase_dayofweek'] = df_clean['purchase_date'].dt.dayofweek  # 0=Segunda, 6=Domingo
df_clean['purchase_quarter'] = df_clean['purchase_date'].dt.quarter
df_clean['is_weekend'] = df_clean['purchase_dayofweek'].isin([5, 6]).astype(int)

Este bloco de código prepara as variáveis categóricas (colunas que contêm texto em vez de números) para poderem ser processadas pelos modelos de Machine Learning.

O que cada linha faz:

cat_cols = [...] -> Cria uma lista com os nomes de todas as colunas do dataset que são categóricas/texto.

df_clean[cat_cols] = df_clean[cat_cols].fillna('Missing') -> Procura por valores em falta (NaN / nulos) dentro dessas 6 colunas e substitui-os pela palavra 'Missing'.

Por que é que isto é fundamental?

⚬ Evita erros na codificação

⚬ Trata a ausência de informação como um padrão

⚬ Cria uma categoria explícita

In [17]:
#Codificação de Variáveis Categóricas
# Tratamento de NAs nas categóricas antes da codificação (atribuindo a classe 'Missing')
cat_cols = ['payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series']
df_clean[cat_cols] = df_clean[cat_cols].fillna('Missing')

O que cada linha faz:

low_cardinality_cols = [...] -> Define a lista das colunas com poucas categorias únicas.

pd.get_dummies(...) -> Função do Pandas para transformar colunas de texto em binárias (0 ou 1).

drop_first=True (Muito Importante) -> Elimina a primeira categoria de cada coluna para evitar a 'Dummy Variable Trap' (multicolinearidade).

dtype=int -> Garante que o resultado das novas colunas seja em números inteiros (0 e 1).

In [19]:
# Baixa Cardinalidade -> One-Hot Encoding
# (payment_type, salesperson, commercial_zone, warehouse, document_series)
low_cardinality_cols = ['payment_type', 'salesperson', 'commercial_zone', 'warehouse', 'document_series']

df_clean = pd.get_dummies(
    df_clean, 
    columns=low_cardinality_cols, 
    drop_first=True,  # Evita a armadilha das variáveis dummy (multicolinearidade)
    dtype=int
)

In [20]:
# Alta Cardinalidade -> Frequency Encoding (ex.: transport_method com 80 categorias)
# Substitui cada categoria pela frequência proporcional/absoluta com que aparece
freq_map = df_clean['transport_method'].value_counts(normalize=True).to_dict()
df_clean['transport_method_freq'] = df_clean['transport_method'].map(freq_map)

A remoção dessas duas colunas específicas no final da fase de pré-processamento responde a duas razões técnicas essenciais:

purchase_date (Coluna Original da Data): Informação já foi extraída, incompatibilidade com o modelo, evitar redundância.

transport_method (Coluna Categórica Original): A categoria de texto foi substituída por um número via Frequency Encoding, evitando erros de treino com texto.

In [21]:
# Remover a coluna original de data e a categórica original de transport_method
df_clean = df_clean.drop(columns=['purchase_date', 'transport_method'])

In [22]:
# Criar a matriz de features (X) pronta para modelação (excluindo ID e Target)
features_to_exclude = ['ID', 'repeat_purchase_90d']
X = df_clean.drop(columns=features_to_exclude)
y = df_clean['repeat_purchase_90d']

In [23]:
# Validação Focada do Bloco
print("=== VALIDAÇÃO DA ENGENHARIA DE ATRIBUTOS E CATEGÓRICAS ===")
print(f"• Novos atributos temporais criados: ['purchase_year', 'purchase_month', 'purchase_day', 'purchase_dayofweek', 'purchase_quarter', 'is_weekend']")
print(f"• 'transport_method' codificado por frequência ('transport_method_freq')")
print(f"• Variáveis com One-Hot Encoding criadas com sucesso (total de colunas atuais: {df_clean.shape[1]})")
print(f"• Coluna 'ID' isolada e pronta para remoção no treino.")
print(f"• Coluna 'random_noise' mantida explicitamente para Feature Selection.")

=== VALIDAÇÃO DA ENGENHARIA DE ATRIBUTOS E CATEGÓRICAS ===
• Novos atributos temporais criados: ['purchase_year', 'purchase_month', 'purchase_day', 'purchase_dayofweek', 'purchase_quarter', 'is_weekend']
• 'transport_method' codificado por frequência ('transport_method_freq')
• Variáveis com One-Hot Encoding criadas com sucesso (total de colunas atuais: 66)
• Coluna 'ID' isolada e pronta para remoção no treino.
• Coluna 'random_noise' mantida explicitamente para Feature Selection.
